# Aula 1 — Um pedido, três papéis com LLM

Vamos passar o pedido P001 por estoque, logística e revisão. Ana, Bruno e Carla usam o mesmo modelo, cada um com seu prompt e seu histórico. Python define a ordem das etapas; o modelo gera as respostas.

A multiplicação continua em uma função pequena, como no Agente matemático. O modelo chama essa ferramenta, e nós conferimos o valor que ela devolveu. Todas as propostas da loja são fictícias.

## Preparar o Colab

Execute a célula e espere a instalação terminar. Depois, siga para o passo 1. As versões são fixas e vêm do PyPI oficial. Se o próprio Colab pedir reinício, use **Runtime → Restart session** e execute desde o início. Use uma sessão CPU; o modelo será acessado na nuvem do Ollama.

In [ ]:
import os, sys, subprocess
from importlib.metadata import version, PackageNotFoundError
IN_COLAB = 'google.colab' in sys.modules or bool(os.environ.get('COLAB_RELEASE_TAG'))
REQUISITOS = {'langchain': '1.4.7', 'langchain-core': '1.7.1', 'pydantic': '2.13.5', 'langchain-ollama': '1.1.0', 'langgraph': '1.2.18', 'ollama': '0.6.3'}
def versao_instalada(nome):
    try: return version(nome)
    except PackageNotFoundError: return None
faltantes = [f'{p}=={v}' for p, v in REQUISITOS.items() if versao_instalada(p) != v]
if IN_COLAB and faltantes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=:all:', '--disable-pip-version-check', '--index-url', 'https://pypi.org/simple', *faltantes], check=True)
elif faltantes: raise RuntimeError('Fora do Colab não instalamos automaticamente. Use o ambiente preparado nas versões indicadas no README.')
print('Python do kernel:', sys.executable, '| Colab:', IN_COLAB)
print('Versões conferidas:', {p: versao_instalada(p) for p in REQUISITOS})

## Antes de começar

No cadeado do Colab, confira o segredo chamado **segredo**, com sua chave do Ollama. Autorize o acesso deste notebook. A chave não deve ser colada nas células nem aparecer nas saídas.

Retomamos o P001 do Módulo 3: duas camisetas a R$ 35 e estoque inicial de oito. Os dados estão preparados neste caderno. Não há pagamento, reserva ou envio real.

## 1. Conectar o modelo

Esta célula importa o LangChain, lê o segredo e configura o Ollama. A primeira pergunta será enviada no passo 3. Não é necessário instalar o Ollama no computador nem baixar o modelo.

In [2]:
import os, json
os.environ['LANGSMITH_TRACING'] = 'false'
os.environ['LANGCHAIN_TRACING_V2'] = 'false'
os.environ['OTEL_SDK_DISABLED'] = 'true'
import dotenv
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
# Local: Ollama na máquina, configurado pelo keys.env da raiz (sem chave de API)
dotenv.load_dotenv(dotenv.find_dotenv('keys.env', usecwd=True))
# temperature=0: o qwen2.5:3b segue regras e chama ferramentas com mais consistência
modelo = ChatOllama(model=os.getenv('OLLAMA_MODEL', 'qwen2.5:3b'),
    base_url=os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434'),
    temperature=0, num_predict=256, validate_model_on_init=False,
    client_kwargs={'timeout': 60})
print('Modelo configurado:', modelo.model, 'em', modelo.base_url)

Modelo configurado: qwen2.5:3b em http://localhost:11434


## Sua vez

Tente criar um sistema multiagente próprio. Utilize alguma das arquiteturas ensinadas.

In [ ]:
# Cozinha do Creiton: dados, tools, agentes e o fluxo do atendimento (rode uma vez; a entrada fica na próxima célula)
gramas_por_pessoa = 150
estoque_gramas = 1000
# estado do atendimento em andamento: entregar_comida lê a permissão e registra as entregas aqui
estado = {'pode_entregar': False, 'entregas_carlos': []}

# Tools: as contas ficam em Python, os agentes só pedem a chamada

@tool
def criar_envelope(id, conteudo, remetente, destinatario):
    """Cria um envelope para direcionar a outros agentes"""
    return {
        "id": id,
        "conteudo": conteudo,
        "remetente": remetente,
        "destinatario": destinatario
    }
@tool
def calcular_porcoes(pessoas: int, gramas_por_pessoa: int) -> int:
    """Calcula o total de gramas necessário para servir todas as pessoas."""
    return pessoas * gramas_por_pessoa

@tool
def calcular_estoque(necessário: float, estoque: float) -> float:
    """Calcula quantos gramas sobram no estoque (estoque - necessário). Resultado negativo indica falta."""
    return estoque-necessário

@tool
def formata_pedido(mesa: str, prato: str, pessoas: int):
    """Formata os pedidos de cada mesa.
       mesa: Mesa do pedido (MXX)
       prato: Prato pedido pela mesa
       pessoas: quantidade de pessoas da mesa
    """
    pedido = {'mesa': mesa, 'prato': prato, 'pessoas': pessoas}
    return pedido

# mesa e prato separados: o 3b manda o pedido como dict e falha na validação de um parâmetro str
@tool
def entregar_comida(mesa: str, prato: str) -> str:
    """Chama o garçom Carlos para entregar o prato à mesa. Só use se a Dona Rosa propôs separar o ingrediente."""
    # o executor confere a permissão: sem estoque, o garçom nem é chamado, mesmo que o revisor peça
    if not estado['pode_entregar']:
        return 'ENTREGA_BLOQUEADA: falta ingrediente, o garçom não foi chamado.'
    entrega = garcom.invoke({'messages': [{'role':'user','content': f'ENTREGAR {prato} na mesa {mesa}'}]},config={'recursion_limit':8})
    texto_entrega = entrega['messages'][-1].content
    estado['entregas_carlos'].append(texto_entrega)  # registra só as vezes em que o Carlos foi de fato chamado
    return texto_entrega

# Agentes: cada um com seu papel, prompt e ferramentas
# Carlos tem duas tarefas: anota o pedido no início e só volta a ser chamado (por entregar_comida) para entregar
# "primeira ação é obrigatoriamente chamar": sem isso o 3b às vezes escreve "Pedido anotado" direto, sem chamar a tool
garcom = create_agent(modelo, tools=[formata_pedido], system_prompt=(
    'Você é Carlos, o garçom do restaurante. '
    'Para todo pedido novo, sua primeira ação é obrigatoriamente chamar formata_pedido com mesa no formato MXX, prato e pessoas. '
    'Depois do resultado, responda "Pedido anotado: <mesa>, <prato>, <pessoas> pessoas." '
    'Exceção: se a mensagem começar com ENTREGAR, não chame ferramentas e responda "Entregue: <prato> na mesa <mesa>." '
    'Responda em português.'))
creiton = create_agent(modelo, tools=[calcular_porcoes], system_prompt=(
    'Você é Creiton, chef de cozinha. Sempre chame calcular_porcoes antes de responder para saber o total de gramas. '
    'Responda em português, em até três linhas, com o total calculado.'))
# formato de resposta fixo: sem ele o 3b confunde o que separar com o que sobra
despensa = create_agent(modelo, tools=[calcular_estoque], system_prompt=(
    'Você é Dona Rosa, da despensa. Sempre chame calcular_estoque antes de responder, '
    'passando necessário=total e estoque=estoque_gramas. A ferramenta devolve o que sobra no estoque. '
    'Se o resultado for negativo, responda no formato: FALTA_INGREDIENTE: faltam <resultado sem o sinal> g. '
    'Se for zero ou positivo, responda no formato: Proposta: separar <total> g; sobram <resultado> g no estoque. '
    'É uma proposta fictícia. Responda em português, em até três linhas.'))
revisor = create_agent(modelo, tools=[entregar_comida], system_prompt=(
    'Você é Paulo, revisor. Confira a conta e o estoque. '
    'Se o parecer da Dona Rosa tiver FALTA_INGREDIENTE, não chame ferramentas: explique a falta e diga que o pedido não será entregue. '
    'Se o parecer for uma Proposta de separar, chame entregar_comida com a mesa e o prato do pedido. '
    'Responda em português, em até três linhas.'))

def resultados_da_tool(resposta, nome):
    """Devolve as mensagens que a tool `nome` gerou dentro da resposta de um agente."""
    return [m for m in resposta['messages'] if isinstance(m, ToolMessage) and m.name == nome]

def atender_pedido(pedido_cliente):
    """Passa o pedido por Carlos, Creiton, Dona Rosa e Paulo e devolve as respostas e os valores das ferramentas."""
    estado['pode_entregar'] = False
    estado['entregas_carlos'] = []

    # Carlos anota o pedido: o dict vem dos argumentos que ele passou a formata_pedido, não do texto dele
    resposta_garcom = garcom.invoke({'messages': [{'role':'user','content': pedido_cliente}]},config={'recursion_limit':8})
    chamadas_pedido = [c for m in resposta_garcom['messages'] for c in (getattr(m, 'tool_calls', None) or [])
                       if c['name'] == 'formata_pedido']
    if not chamadas_pedido:
        raise RuntimeError('Carlos não anotou o pedido com formata_pedido. Resposta dele: ' + resposta_garcom['messages'][-1].content)
    pedido_mesa = chamadas_pedido[-1]['args']
    contexto_cozinha = json.dumps({'pedido': pedido_mesa, 'gramas_por_pessoa': gramas_por_pessoa,
                                   'estoque_gramas': estoque_gramas}, ensure_ascii=False)

    # Creiton calcula o total; o total vem do ToolMessage (valor exato da ferramenta), não do texto dele
    resposta_creiton = creiton.invoke({'messages': [{'role':'user','content': contexto_cozinha}]},config={'recursion_limit':8})
    texto_creiton = resposta_creiton['messages'][-1].content
    ferramentas_cozinha = resultados_da_tool(resposta_creiton, 'calcular_porcoes')
    if not ferramentas_cozinha:
        raise RuntimeError('Creiton não calculou o total com calcular_porcoes. Resposta dele: ' + texto_creiton)
    total_observado = int(float(ferramentas_cozinha[-1].content))

    # Dona Rosa recebe só os números: com o texto do Creiton junto, o 3b copia o parecer em vez de aplicar a regra
    mensagem_despensa = json.dumps({'total': total_observado, 'estoque_gramas': estoque_gramas}, ensure_ascii=False)
    resposta_despensa = despensa.invoke({'messages': [{'role':'user','content': mensagem_despensa}]},config={'recursion_limit':8})
    texto_despensa = resposta_despensa['messages'][-1].content
    ferramentas_despensa = resultados_da_tool(resposta_despensa, 'calcular_estoque')
    if not ferramentas_despensa:
        raise RuntimeError('Dona Rosa não calculou o estoque com calcular_estoque. Resposta dela: ' + texto_despensa)
    sobra_observada = float(ferramentas_despensa[-1].content)
    # a permissão de entrega vem do valor da ferramenta, não do texto da Dona Rosa
    estado['pode_entregar'] = sobra_observada >= 0

    # Paulo revisa com o pedido, os números e os dois pareceres; se estiver tudo ok, chama o garçom
    mensagem_revisor = (contexto_cozinha + '\n' + mensagem_despensa
                        + '\nParecer do Creiton:\n' + texto_creiton
                        + '\nParecer da Dona Rosa:\n' + texto_despensa)
    resposta_revisor = revisor.invoke({'messages': [{'role':'user','content': mensagem_revisor}]},config={'recursion_limit':8})

    return {
        'pedido_mesa': pedido_mesa,
        'carlos': resposta_garcom['messages'][-1].content,
        'creiton': texto_creiton,
        'dona_rosa': texto_despensa,
        'paulo': resposta_revisor['messages'][-1].content,
        # pedir a entrega (Paulo chamar a tool) é diferente de o Carlos ser chamado (a tool liberar a entrega)
        'pedidos_entrega': [m.content for m in resultados_da_tool(resposta_revisor, 'entregar_comida')],
        'entregas_carlos': list(estado['entregas_carlos']),
        'total_observado': total_observado,
        'sobra_observada': sobra_observada,
    }

def conferir(r):
    """Confere o atendimento pelos valores das ferramentas, não pelo texto dos agentes; devolve a lista de problemas."""
    problemas = []
    if r['total_observado'] != r['pedido_mesa']['pessoas'] * gramas_por_pessoa:
        problemas.append('o total não confere')
    if r['sobra_observada'] != estoque_gramas - r['total_observado']:
        problemas.append('a sobra do estoque não confere')
    if r['total_observado'] <= estoque_gramas:
        if not r['entregas_carlos']:
            problemas.append('com estoque, o Paulo deveria ter chamado o garçom para entregar')
    else:
        if 'FALTA_INGREDIENTE' not in r['dona_rosa'].upper():
            problemas.append('Dona Rosa deveria informar FALTA_INGREDIENTE')
        if r['entregas_carlos']:
            problemas.append('sem estoque, o garçom não poderia ter sido chamado')
    return problemas

print('Cozinha pronta: rode a próxima célula para fazer pedidos.')

Cozinha pronta: rode a próxima célula para fazer pedidos.


In [4]:
# Entrada e saída: digite um pedido por vez (ex.: "Mesa 4 quer arroz de carreteiro para 5 pessoas"); Enter vazio encerra
while True:
    pedido_cliente = input('Pedido da mesa (Enter vazio para sair): ').strip()
    if not pedido_cliente:
        break
    print('\nPedido recebido:', pedido_cliente)
    try:
        r = atender_pedido(pedido_cliente)
    except RuntimeError as erro:
        print('Atendimento interrompido:', erro)
        continue

    print('Carlos — garçom:', r['carlos'])
    print('Creiton — cozinha:', r['creiton'])
    print('Dona Rosa — despensa:', r['dona_rosa'])
    print('Paulo — revisão:', r['paulo'])
    for texto in r['pedidos_entrega']:
        print('Paulo pediu entregar_comida →', texto)
    print('Carlos — entrega:', r['entregas_carlos'][-1] if r['entregas_carlos'] else 'não foi chamado para entregar.')

    print('Ferramentas: total', r['total_observado'], 'g | sobra', r['sobra_observada'], 'g | Carlos chamado para entregar:', len(r['entregas_carlos']))
    problemas = conferir(r)
    print('Conferência:', 'ok' if not problemas else 'PROBLEMA — ' + '; '.join(problemas))


Pedido recebido: mesa 02 tambaqui pra 2
Carlos — garçom: Pedido anotado: 02, tambaqui, 2 pessoas.
Creiton — cozinha: Para atender a mesa 02, que pediu para o prato de tambaqui para duas pessoas, serão necessárias 300 gramas. Com base no estoque atual de 1000 gramas, teremos 700 gramas suficientes para atender a essa mesa.
Dona Rosa — despensa: Proposta: separar 300 g; sobram 700 g no estoque.
Paulo — revisão: Parecer da Dona Rosa: Proposta de separar 300g; sobram 700g no estoque. Pedido para mesa 02 será entregue.
Paulo pediu entregar_comida → Entregue: tambaqui na mesa 02.
Carlos — entrega: Entregue: tambaqui na mesa 02.
Ferramentas: total 300 g | sobra 700.0 g | Carlos chamado para entregar: 1
Conferência: ok


## Para consultar

- [Agentes com create_agent — LangChain](https://docs.langchain.com/oss/python/langchain/agents)
- [ChatOllama — LangChain](https://docs.langchain.com/oss/python/integrations/chat/ollama)
- [API na nuvem — Ollama](https://docs.ollama.com/cloud)

Fontes conferidas em 29/10/2026. A execução depende da chave, do acesso ao modelo e da disponibilidade do Ollama.